# Property ML Lab: from missing measurements to honest uncertainty

A beginner walkthrough inside **Sprint Lead Generation**. We predict historical Ames, Iowa sale prices, not current Georgia prices or lead conversion. The source has real observed sale prices and real missing measurements. No artificial labels are created.

Run the setup in `README.md` first. This notebook uses the same small functions as the app, and every code cell below has been executed. The split seed, eight features, Ridge penalty of 10, and 90% interval level are fixed before inspecting test results.


In [1]:
from pathlib import Path
import sys
import numpy as np
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd()
if not (ROOT / 'ml_lab').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'ml_lab').is_dir(), 'Open this notebook from the repository.'
sys.path.insert(0, str(ROOT))
from ml_lab.core import (FEATURES, load_dataset, split_dataset, make_pipeline,
                         bootstrap_mean_ci, conformal_half_width, train_experiment)
from ml_lab.data import download_ames
DATA = ROOT / 'data/ml/AmesHousing.tsv'
if not DATA.exists():
    download_ames(DATA)
dataset = load_dataset(DATA)
print(f'{len(dataset.y):,} labeled sales, {dataset.X.shape[1]} input features')
print('Source SHA-256:', dataset.sha256)


2,930 labeled sales, 8 input features
Source SHA-256: 6cfe6cb525ba437de428653a1040e2aed7d696640bf75203786a6d7a0e67cfcc


## 1. Look at what is missing

A missing measurement is unknown; zero is an observed value. In particular, a garage area of zero must stay zero. We never fill in missing sale-price labels. The eight selected fields are numeric, so they avoid the source's categorical `NA` codes meaning things such as “no fireplace.”


In [2]:
for index, feature in enumerate(FEATURES):
    count = int(np.isnan(dataset.X[:, index]).sum())
    print(f'{feature.label:28} {count:4} missing ({count / len(dataset.y):5.1%})')
print('Rows with any missing input:', int(np.isnan(dataset.X).any(axis=1).sum()))


Lot frontage                  490 missing (16.7%)
Above-ground living area        0 missing ( 0.0%)
Lot area                        0 missing ( 0.0%)
Overall quality                 0 missing ( 0.0%)
Year built                      0 missing ( 0.0%)
Full bathrooms                  0 missing ( 0.0%)
Above-ground bedrooms           0 missing ( 0.0%)
Garage area                     1 missing ( 0.0%)
Rows with any missing input: 491


## 2. Split before learning anything

Training rows teach the model and preprocessing. Calibration rows set the prediction interval. Test rows measure final performance. Property IDs are checked across the splits but are never used as model inputs. If repeated properties are supplied, the implementation keeps each property in one partition.


In [3]:
split = split_dataset(dataset)
train, calibration, test = (split[name] for name in ('train', 'calibration', 'test'))
for name, rows in split.items():
    print(f'{name:12} {len(rows):,} rows')
for a, b in [('train', 'calibration'), ('train', 'test'), ('calibration', 'test')]:
    assert set(dataset.property_ids[split[a]]).isdisjoint(dataset.property_ids[split[b]])


train        1,758 rows
calibration  586 rows
test         586 rows


## 3. Train a simple pipeline

`SimpleImputer` fills unknown inputs with training medians and adds missingness flags. `StandardScaler` makes feature scales comparable. `Ridge` learns a linear formula with a penalty on large coefficients. The source helper contains these three steps explicitly.

The baseline predicts the same median training price for every home. We compare both models on the same test cases. No preprocessing is fitted on calibration or test data.


In [4]:
model = make_pipeline()
print(model)
model.fit(dataset.X[train], dataset.y[train])
training_medians = model.named_steps['imputer'].statistics_
assert np.allclose(training_medians, np.nanmedian(dataset.X[train], axis=0))
for feature, median in zip(FEATURES, training_medians):
    print(f'{feature.label:28} training median = {median:g}')
baseline_price = float(np.median(dataset.y[train]))
print(f'Baseline prediction for every home: ${baseline_price:,.0f}')


Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, keep_empty_features=True,
                               strategy='median')),
                ('scaler', StandardScaler()), ('ridge', Ridge(alpha=10.0))])
Lot frontage                 training median = 67
Above-ground living area     training median = 1440
Lot area                     training median = 9350
Overall quality              training median = 6
Year built                   training median = 1972
Full bathrooms               training median = 2
Above-ground bedrooms        training median = 3
Garage area                  training median = 480
Baseline prediction for every home: $160,100


## 4. Calibrate an interval before opening the test results

Compute absolute prediction errors on the separate calibration set. For a 90% split-conformal interval, use the `ceil((m + 1) × 0.90)`th smallest residual, where `m` is the number of calibration cases. The finite-sample correction matters.

This produces one shared radius. Under exchangeability, it targets coverage across cases overall; it does not guarantee 90% coverage for each kind of home, or for invented input combinations.


In [5]:
calibration_predictions = model.predict(dataset.X[calibration])
calibration_errors = np.abs(dataset.y[calibration] - calibration_predictions)
radius, rank = conformal_half_width(calibration_errors, level=0.90)
print(f'Calibration residual rank: {rank} of {len(calibration)}')
print(f'90% prediction interval: predicted price ± ${radius:,.0f}')


Calibration residual rank: 529 of 586
90% prediction interval: predicted price ± $46,924


## 5. Evaluate once, with confidence intervals

MAE is the mean absolute error, in dollars. The bootstrap samples held-out errors with replacement 2,000 times, while keeping the fitted model fixed. Its 95% interval describes uncertainty in average error on comparable cases, conditional on this trained model. It does not include variability from retraining or market change.

Resampling the *paired error differences* gives a fair confidence interval for improvement because both models are measured on the same homes.


In [6]:
predictions = model.predict(dataset.X[test])
actual = dataset.y[test]
ridge_errors = np.abs(actual - predictions)
baseline_errors = np.abs(actual - baseline_price)
for name, errors in [('Median baseline', baseline_errors), ('Ridge', ridge_errors)]:
    lo, hi = bootstrap_mean_ci(errors)
    print(f'{name:16} MAE ${errors.mean():,.0f}; 95% CI [${lo:,.0f}, ${hi:,.0f}]')
improvement = baseline_errors - ridge_errors
lo, hi = bootstrap_mean_ci(improvement)
print(f'Paired MAE improvement: ${improvement.mean():,.0f}; 95% CI [${lo:,.0f}, ${hi:,.0f}]')
covered = np.abs(actual - predictions) <= radius
print(f'Measured prediction-interval coverage: {covered.sum()}/{len(test)} = {covered.mean():.1%}')


Median baseline  MAE $58,677; 95% CI [$53,799, $63,852]
Ridge            MAE $22,991; 95% CI [$21,071, $25,008]
Paired MAE improvement: $35,686; 95% CI [$31,708, $39,652]
Measured prediction-interval coverage: 536/586 = 91.5%


## 6. Trace one missing value through the app

This example deliberately leaves frontage unknown. The returned imputation record reveals the exact training median used. A changed missingness pattern may differ from the evaluation data; a successful prediction is not proof that the interval remains reliable for that invented profile.


In [7]:
experiment = train_experiment(DATA)
profile = {feature.key: float(training_medians[i]) for i, feature in enumerate(FEATURES)}
profile['lot_frontage'] = None
result = experiment.predict(profile)
print(f"Estimated historical price: ${result['prediction']:,.0f}")
print(f"90% prediction interval: [${result['lower']:,.0f}, ${result['upper']:,.0f}]")
print('Filled inputs:', result['imputed_fields'])
for warning in result['warnings']:
    print('Note:', warning)
report_ridge = next(m for m in experiment.report['models'] if m['key'] == 'ridge')
assert np.isclose(report_ridge['mae'], mean_absolute_error(actual, predictions))
print('Notebook and app report agree on Ridge MAE.')


Estimated historical price: $167,555
90% prediction interval: [$120,631, $214,479]
Filled inputs: [{'key': 'lot_frontage', 'label': 'Lot frontage', 'value': 67.0}]
Notebook and app report agree on Ridge MAE.


## What I would explain in an interview

- Why imputation belongs inside the training pipeline.
- Why the baseline is useful even when its error is high.
- Why a confidence interval for mean error is different from a prediction interval for a sale price.
- Why missing values do not justify inventing labels or treating unlabeled leads as unsuccessful.
- Why these historical results do not establish current Georgia accuracy or seller conversion.

Try calculating MAE by hand for three homes, tracing one filled value, and explaining the calibration rank without reading the code. Further studies should use training-only model selection and a fresh final evaluation set rather than repeatedly tuning to this test set.

### Sources

[De Cock's original study](https://jse.amstat.org/v19n3/decock.pdf), [scikit-learn leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html), [Ridge regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [bootstrap reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html), and [Angelopoulos & Bates on conformal prediction](https://arxiv.org/abs/2107.07511). See `docs/ML_RESEARCH.md` for the detailed methodological limitations and attribution.
